# 01 — Shapefile Processing & Area Computation

Loads TIGER/Line shapefiles for the 5 project states, extracts census block polygon geometries, computes land areas, builds the CB adjacency table, and prepares the spatial foundation for density calculations and spatial features.

**Input Files:**
- `data/Shape Files/tl_2025_06_tabblock20/` (California)
- `data/Shape Files/tl_2025_13_tabblock20/` (Georgia)
- `data/Shape Files/tl_2025_17_tabblock20/` (Illinois)
- `data/Shape Files/tl_2025_36_tabblock20/` (New York)
- `data/Shape Files/tl_2025_48_tabblock20/` (Texas)

**Outputs:**
- `output/geometry.geoparquet` — Full CB geometry table with area
- `output/adjacency.parquet` — CB neighbor pairs (spatial touches)
- `output/area_sq_km.csv` — Lightweight join table (cb_fips, area_sq_km)

**Key Shapefile Columns:**
- `GEOID20` — 15-digit Census Block FIPS code (join key)
- `ALAND20` — Land area in square meters (Census Bureau official computation)
- `AWATER20` — Water area in square meters
- `geometry` — Block polygon

## 1. Setup & Imports

In [1]:
import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.4f}'.format)

# --- File Paths ---
SHAPEFILE_DIR = '../../data/Shape Files'
OUTPUT_DIR = '../../output'

# State FIPS → shapefile directory mapping
STATE_FILES = {
    '06': os.path.join(SHAPEFILE_DIR, 'tl_2025_06_tabblock20', 'tl_2025_06_tabblock20.shp'),
    '13': os.path.join(SHAPEFILE_DIR, 'tl_2025_13_tabblock20', 'tl_2025_13_tabblock20.shp'),
    '17': os.path.join(SHAPEFILE_DIR, 'tl_2025_17_tabblock20', 'tl_2025_17_tabblock20.shp'),
    '36': os.path.join(SHAPEFILE_DIR, 'tl_2025_36_tabblock20', 'tl_2025_36_tabblock20.shp'),
    '48': os.path.join(SHAPEFILE_DIR, 'tl_2025_48_tabblock20', 'tl_2025_48_tabblock20.shp'),
}

STATE_NAMES = {'06': 'CA', '13': 'GA', '17': 'IL', '36': 'NY', '48': 'TX'}

# Columns to retain from shapefiles (drop the rest to save memory)
KEEP_COLS = ['GEOID20', 'STATEFP20', 'COUNTYFP20', 'ALAND20', 'AWATER20',
             'INTPTLAT20', 'INTPTLON20', 'geometry']

print("File paths configured.")
print(f"States: {list(STATE_NAMES.values())}")

File paths configured.
States: ['CA', 'GA', 'IL', 'NY', 'TX']


## 2. Load & Process Shapefiles

Load each state's TIGER/Line shapefile, retain only essential columns, and compute `area_sq_km` from the Census Bureau's official `ALAND20` field (land area in square meters).

**CRS:** Shapefiles arrive in EPSG:4269 (NAD83). This is retained as-is — it's functionally equivalent to WGS84 for web mapping and avoids reprojection overhead.

**Area:** `ALAND20` is the Census Bureau's pre-computed **land area** (excludes water). This is more accurate than reprojecting and computing from the polygon geometry, and it's the authoritative value used in all Census products.

In [2]:
frames = []

for fips, filepath in STATE_FILES.items():
    state_name = STATE_NAMES[fips]
    t0 = time.time()
    print(f"Loading {state_name} ({fips})...", end=' ', flush=True)
    
    gdf = gpd.read_file(filepath, columns=KEEP_COLS)
    
    elapsed = time.time() - t0
    print(f"{len(gdf):,} blocks in {elapsed:.1f}s")
    frames.append(gdf)

### Concatenate all states
all_blocks = pd.concat(frames, ignore_index=True)
del frames  # free memory

### Rename GEOID20 → cb_fips
all_blocks = all_blocks.rename(columns={'GEOID20': 'cb_fips', 'STATEFP20': 'state_fips',
                                         'COUNTYFP20': 'county_fips'})

### Computing the shape from the geometry column
shapes_gpd = gpd.GeoDataFrame(all_blocks, geometry='geometry')
shapes_gpd = shapes_gpd.to_crs(epsg=3857) ### Converting to Web Mercator for accurate area calculations

### Compute area_sq_km from ALAND20 (square meters → square kilometers)
# all_blocks['area_sq_km'] = all_blocks['ALAND20'] / 1_000_000

### Geopandas function to compute area in square kilometers
all_blocks['area_sq_km'] = shapes_gpd['geometry'].area / 10**6

### Deleting shapes_gpd to free memory
del shapes_gpd

### Verify CRS
print(f"\nCRS: {all_blocks.crs}")
print(f"Total blocks: {len(all_blocks):,}")
print(f"States: {sorted(all_blocks['state_fips'].unique())}")
all_blocks.head()

Loading CA (06)... 519,723 blocks in 12.2s
Loading GA (13)... 232,717 blocks in 4.9s
Loading IL (17)... 369,978 blocks in 7.5s
Loading NY (36)... 288,819 blocks in 7.5s
Loading TX (48)... 668,757 blocks in 17.8s

CRS: EPSG:4269
Total blocks: 2,079,994
States: ['06', '13', '17', '36', '48']


,state_fips,county_fips,cb_fips,ALAND20,AWATER20,INTPTLAT20,INTPTLON20,geometry,area_sq_km
0,06,021,060210102002055,1063280,7323,+39.7757057,-122.2005037,"POLYGON ((-122.20475 39.77111, -122.20466 39.7...",1.8147
1,06,021,060210103001172,0,44454,+39.4206747,-122.5288043,"POLYGON ((-122.53008 39.42053, -122.52996 39.4...",0.0746
2,06,021,060210103001081,1329328,5260,+39.5846150,-122.4083926,"POLYGON ((-122.44617 39.58683, -122.44591 39.5...",2.2498
3,06,021,060210105013037,0,264797,+39.6658034,-122.0258944,"POLYGON ((-122.0284 39.66156, -122.02603 39.66...",0.4476
4,06,021,060210105013028,17833,143456,+39.7153617,-122.0106023,"POLYGON ((-122.0216 39.71604, -122.0198 39.716...",0.2729


## 3. Summary Statistics & Edge Cases

In [5]:
### Per-state block counts
print("=== Per-State Block Counts ===")
for fips in sorted(STATE_NAMES.keys()):
    name = STATE_NAMES[fips]
    n = (all_blocks['state_fips'] == fips).sum()
    print(f"  {name} ({fips}): {n:,} blocks")
print(f"  Total: {len(all_blocks):,}")

### CB FIPS integrity
null_fips = all_blocks['cb_fips'].isna().sum()
bad_len = (all_blocks['cb_fips'].str.len() != 15).sum()
dupes = all_blocks['cb_fips'].duplicated().sum()
print(f"\n[{'PASS' if null_fips == 0 else 'FAIL'}] Null cb_fips: {null_fips}")
print(f"[{'PASS' if bad_len == 0 else 'FAIL'}] Invalid cb_fips length: {bad_len}")
print(f"[{'PASS' if dupes == 0 else 'FAIL'}] Duplicate cb_fips: {dupes}")

### Area distribution
print(f"\n--- Area (sq km) Distribution ---")
print(all_blocks['area_sq_km'].describe())

### Edge cases: zero land area (water-only blocks)
zero_area = (all_blocks['ALAND20'] == 0).sum()
tiny_area = (all_blocks['area_sq_km'] < 0.001).sum()
water_only = ((all_blocks['ALAND20'] == 0) & (all_blocks['AWATER20'] > 0)).sum()
print(f"\n[INFO] Zero land area blocks: {zero_area:,}")
print(f"[INFO] Water-only blocks (ALAND=0, AWATER>0): {water_only:,}")
print(f"[INFO] Tiny area blocks (<0.001 sq km): {tiny_area:,}")
print("  → These are kept (may appear in FCC data). Densities capped in feature engineering.")

=== Per-State Block Counts ===
  CA (06): 519,723 blocks
  GA (13): 232,717 blocks
  IL (17): 369,978 blocks
  NY (36): 288,819 blocks
  TX (48): 668,757 blocks
  Total: 2,079,994

[PASS] Null cb_fips: 0
[PASS] Invalid cb_fips length: 0
[PASS] Duplicate cb_fips: 0

--- Area (sq km) Distribution ---
count   2,079,994.0000
mean            1.1400
std             9.7206
min             0.0000
25%             0.0207
50%             0.0507
75%             0.3465
max         3,701.6703
Name: area_sq_km, dtype: float64

[INFO] Zero land area blocks: 41,758
[INFO] Water-only blocks (ALAND=0, AWATER>0): 41,758
[INFO] Tiny area blocks (<0.001 sq km): 2,320
  → These are kept (may appear in FCC data). Densities capped in feature engineering.


## 4. Build CB Adjacency Table

For each census block, identify all neighboring blocks that share a boundary (spatial `touches` predicate). This powers the **spatial contagion features** in Phase 3 — the percentage of a block's neighbors that have fiber.

Processing is done **state by state** to manage memory. Since none of the 5 project states share borders, no cross-state neighbors are missed.

In [6]:
adjacency_frames = []

for fips in sorted(STATE_NAMES.keys()):
    state_name = STATE_NAMES[fips]
    t0 = time.time()
    print(f"Computing adjacency for {state_name} ({fips})...", end=' ', flush=True)
    
    # Extract this state's blocks with just cb_fips and geometry (keep default integer index)
    state_mask = all_blocks['state_fips'] == fips
    state_gdf = all_blocks.loc[state_mask, ['cb_fips', 'geometry']].copy()
    
    # Spatial self-join: find all pairs of blocks that touch
    # With overlapping column name 'cb_fips', sjoin suffixes them _left / _right
    neighbors = gpd.sjoin(state_gdf, state_gdf, predicate='touches', how='inner')
    
    # Extract (cb_fips, neighbor_fips) pairs from the suffixed columns
    adj = neighbors[['cb_fips_left', 'cb_fips_right']].rename(
        columns={'cb_fips_left': 'cb_fips', 'cb_fips_right': 'neighbor_fips'}
    )
    
    # Remove self-pairs (a block touching itself)
    adj = adj[adj['cb_fips'] != adj['neighbor_fips']]
    
    elapsed = time.time() - t0
    print(f"{len(adj):,} neighbor pairs in {elapsed:.1f}s")
    adjacency_frames.append(adj)
    
    del state_gdf, neighbors, adj  # free memory

### Combine all states
adjacency = pd.concat(adjacency_frames, ignore_index=True)
del adjacency_frames

print(f"\nTotal adjacency pairs: {len(adjacency):,}")
print(f"Unique blocks with neighbors: {adjacency['cb_fips'].nunique():,}")

### Neighbor count distribution
neighbor_counts = adjacency.groupby('cb_fips').size()
print(f"\n--- Neighbors per Block ---")
print(neighbor_counts.describe())
adjacency.head(10)

Computing adjacency for CA (06)... 3,448,838 neighbor pairs in 251.0s
Computing adjacency for GA (13)... 1,516,142 neighbor pairs in 121.2s
Computing adjacency for IL (17)... 2,558,432 neighbor pairs in 117.8s
Computing adjacency for NY (36)... 1,958,298 neighbor pairs in 123.9s
Computing adjacency for TX (48)... 4,472,120 neighbor pairs in 299.1s

Total adjacency pairs: 13,953,830
Unique blocks with neighbors: 2,079,992

--- Neighbors per Block ---
count   2,079,992.0000
mean            6.7086
std             3.0570
min             1.0000
25%             5.0000
50%             6.0000
75%             8.0000
max            80.0000
dtype: float64


,cb_fips,neighbor_fips
0,060210102002055,060210101012000
1,060210102002055,060210102002057
2,060210102002055,060210102002056
3,060210102002055,060210101012001
4,060210102002055,060210102002085
5,060210102002055,060210102002084
6,060210102002055,060210102002050
7,060210102002055,060210102002054
8,060210102002055,060210102002052
9,060210102002055,060210102002053


## 5. Validation

In [7]:
print("=" * 60)
print("VALIDATION REPORT")
print("=" * 60)

# 1. Geometry validity
invalid_geom = (~all_blocks.geometry.is_valid).sum()
print(f"\n[{'PASS' if invalid_geom == 0 else 'WARN'}] Invalid geometries: {invalid_geom}")

# 2. Empty geometries
empty_geom = all_blocks.geometry.is_empty.sum()
print(f"[{'PASS' if empty_geom == 0 else 'WARN'}] Empty geometries: {empty_geom}")

# 3. Adjacency symmetry check (if A touches B, B should touch A)
adj_set = set(zip(adjacency['cb_fips'], adjacency['neighbor_fips']))
reverse_set = set(zip(adjacency['neighbor_fips'], adjacency['cb_fips']))
asymmetric = len(adj_set - reverse_set)
print(f"[{'PASS' if asymmetric == 0 else 'WARN'}] Asymmetric adjacency pairs: {asymmetric}")

# 4. Self-referencing in adjacency
self_refs = (adjacency['cb_fips'] == adjacency['neighbor_fips']).sum()
print(f"[{'PASS' if self_refs == 0 else 'WARN'}] Self-referencing pairs: {self_refs}")

# 5. Blocks with zero neighbors
blocks_with_neighbors = set(adjacency['cb_fips'].unique())
all_block_fips = set(all_blocks['cb_fips'].unique())
isolated_blocks = all_block_fips - blocks_with_neighbors
print(f"[INFO] Blocks with zero neighbors (isolated): {len(isolated_blocks):,}")

# 6. Cross-check block counts
print(f"\n--- Cross-Check ---")
print(f"Blocks in geometry table: {len(all_blocks):,}")
print(f"Blocks in adjacency table: {adjacency['cb_fips'].nunique():,}")
print(f"Isolated blocks: {len(isolated_blocks):,}")

VALIDATION REPORT

[PASS] Invalid geometries: 0
[PASS] Empty geometries: 0
[PASS] Asymmetric adjacency pairs: 0
[PASS] Self-referencing pairs: 0
[INFO] Blocks with zero neighbors (isolated): 2

--- Cross-Check ---
Blocks in geometry table: 2,079,994
Blocks in adjacency table: 2,079,992
Isolated blocks: 2


## 6. Export

Three outputs:
1. **`geometry.geoparquet`** — Full CB geometry table for PostGIS loading (Phase 6)
2. **`adjacency.parquet`** — CB neighbor pairs for spatial features (Phase 3)
3. **`area_sq_km.csv`** — Lightweight area table for density computation (Phase 2)

In [8]:
### 1. GeoParquet — full geometry table
geo_cols = ['cb_fips', 'state_fips', 'county_fips', 'ALAND20', 'AWATER20',
            'INTPTLAT20', 'INTPTLON20', 'area_sq_km', 'geometry']
geo_out = all_blocks[geo_cols].copy()

geoparquet_path = os.path.join(OUTPUT_DIR, 'geometry.geoparquet')
geo_out.to_parquet(geoparquet_path, index=False)

gp_mb = os.path.getsize(geoparquet_path) / (1024 * 1024)
print(f"GeoParquet: {geoparquet_path} ({gp_mb:.1f} MB)")

### 2. Adjacency Parquet
adjacency_path = os.path.join(OUTPUT_DIR, 'adjacency.parquet')
adjacency.to_parquet(adjacency_path, index=False, engine='pyarrow')

adj_mb = os.path.getsize(adjacency_path) / (1024 * 1024)
print(f"Adjacency:  {adjacency_path} ({adj_mb:.1f} MB, {len(adjacency):,} pairs)")

### 3. Area CSV — lightweight join table for notebook 02
area_df = all_blocks[['cb_fips', 'area_sq_km']].copy()
area_csv_path = os.path.join(OUTPUT_DIR, 'area_sq_km.csv')
area_df.to_csv(area_csv_path, index=False)

area_mb = os.path.getsize(area_csv_path) / (1024 * 1024)
print(f"Area CSV:   {area_csv_path} ({area_mb:.1f} MB, {len(area_df):,} rows)")

### Quick verify
print(f"\n--- Verify area_sq_km.csv ---")
verify = pd.read_csv(area_csv_path, dtype={'cb_fips': str}, nrows=5)
print(verify)

GeoParquet: ../../output/geometry.geoparquet (1828.2 MB)
Adjacency:  ../../output/adjacency.parquet (104.7 MB, 13,953,830 pairs)
Area CSV:   ../../output/area_sq_km.csv (71.3 MB, 2,079,994 rows)

--- Verify area_sq_km.csv ---
           cb_fips  area_sq_km
0  060210102002055      1.8147
1  060210103001172      0.0746
2  060210103001081      2.2498
3  060210105013037      0.4476
4  060210105013028      0.2729
